<a href="https://colab.research.google.com/github/JoaoMAraujoJr/Resolu-es-Atividades-Aprendizado-de-Maquina-Supervisionado/blob/main/exerccios_da_unidade_i.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercícios de Clustering

## K-Means

### Exercício 1: Análise e Seleção de Features

Carregue o dataset `wine` e use a função `seaborn.pairplot` para visualizar as relações entre as features. Analise o gráfico e escolha o par de features que você acredita que melhor separa os 3 grupos. Plote um gráfico de dispersão apenas com o par selecionado.

In [ ]:
from sklearn.datasets import load_wine
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import mode

wine = load_wine()
X = wine.data
y_true = wine.target

df = pd.DataFrame(X, columns= wine.feature_names)
df["class"] = y_true

sns.pairplot(df, hue="class")
plt.show()

plt.figure(figsize=(8,6))

plt.scatter(
    df["od280/od315_of_diluted_wines"],
    df["proline"],
    c='gray',
    cmap="viridis",
    edgecolor="k",
    s=50
)

plt.xlabel("OD280/OD315 of diluted wines")
plt.ylabel("Proline")
plt.title("Dataset Wine - Relação entre proline e OD280/OD315 de vinhos diluidos")

plt.grid(True)
plt.show()

### Exercício 2: Encontrando o K Ótimo

Aplique o Método do Cotovelo nas duas features que você escolheu. Calcule e plote a inércia (WCSS) para K de 1 a 10. Com base no seu gráfico, qual parece ser o número ideal de clusters?

In [ ]:
class KMeans:
    def __init__(self, n_clusters=3, max_iter=100, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.centroids = None
        self.labels = None

    def _initialize_centroids(self, X):
        """
        Inicializa os centróides selecionando K pontos aleatórios do dataset.
        """
        np.random.seed(self.random_state)
        n_samples = X.shape[0]
        random_indices = np.random.choice(n_samples, self.n_clusters, replace=False)
        self.centroids = X[random_indices]

    def _assign_clusters(self, X):
        """
        Atribui cada ponto de dado ao centróide mais próximo.
        """
        n_samples = X.shape[0]
        distances = np.zeros((n_samples, self.n_clusters))

        for i, centroid in enumerate(self.centroids):
            distances[:, i] = np.sum((X - centroid)**2, axis=1)

        self.labels = np.argmin(distances, axis=1)

    def _update_centroids(self, X):
        """
        Atualiza a posição de cada centróide com base na média dos pontos atribuídos a ele.
        """
        new_centroids = np.zeros((self.n_clusters, X.shape[1]))

        for i in range(self.n_clusters):
            cluster_points = X[self.labels == i]
            if len(cluster_points) > 0:
                new_centroids[i] = np.mean(cluster_points, axis=0)
            else:
                new_centroids[i] = self.centroids[i]

        self.centroids = new_centroids

    def fit(self, X):
        """
        Executa o algoritmo K-Means.
        """
        self._initialize_centroids(X)

        for _ in range(self.max_iter):
            old_centroids = self.centroids.copy()
            self._assign_clusters(X)
            self._update_centroids(X)
            if np.allclose(old_centroids, self.centroids):
                break

    def predict(self, X):
        """
        Atribui clusters para novos dados com base nos centróides aprendidos.
        """
        distances = np.zeros((X.shape[0], self.n_clusters))
        for i, centroid in enumerate(self.centroids):
            distances[:, i] = np.sum((X - centroid)**2, axis=1)

        return np.argmin(distances, axis=1)

In [ ]:
k_range = range(1, 11)
inertias = []
clustering_results = []

X_wine = df[["od280/od315_of_diluted_wines", "proline"]].values
for k in k_range:
    model = KMeans(n_clusters=k, max_iter=150, random_state=42)
    model.fit(X_wine)

    # Calcular a inércia (WCSS)
    current_inertia = 0
    for i in range(k):
        # Seleciona os pontos pertencentes ao cluster i
        cluster_points = X_wine[model.labels == i]
        # Calcula a soma das distâncias quadradas ao centróide do cluster i
        current_inertia += np.sum((cluster_points - model.centroids[i])**2)

    inertias.append(current_inertia)
    clustering_results.append({'labels': model.labels, 'centroids': model.centroids})

# Plotar o gráfico do cotovelo
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertias, 'bo-')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia (WCSS)')
plt.title('Método do Cotovelo para Encontrar o K Ótimo')
plt.xticks(k_range)
plt.grid(True)
plt.show()

### Exercício 3: Clusterização e Avaliação

Use o K encontrado na tarefa anterior para treinar seu modelo `KMeans`. Crie um gráfico com dois subplots: um mostrando os clusters encontrados pelo algoritmo e outro mostrando os dados com os rótulos reais para comparação. Por fim, calcule a taxa de acertos e comente o resultado.

In [ ]:
model = KMeans(n_clusters=3, max_iter=150, random_state=42)
model.fit(X_wine)
y_pred = model.predict(X_wine)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Modelo
scatter1 = ax1.scatter(X_wine[:, 0], X_wine[:, 1], c=y_pred, cmap='viridis', edgecolor='k')
ax1.set_title("Clusters do seu KMeans")
ax1.set_xlabel("OD280/OD315")
ax1.set_ylabel("Proline")

# Reais
scatter2 = ax2.scatter(X_wine[:, 0], X_wine[:, 1], c=y_true, cmap='plasma', edgecolor='k')
ax2.set_title("Rótulos Reais")
ax2.set_xlabel("OD280/OD315")
ax2.set_ylabel("Proline")

plt.show()

correct_predictions = 0
n_samples = X_wine.shape[0]

for i in range(model.n_clusters):
    mask = (y_pred == i)
    dominant_label = mode(y_true[mask], keepdims=True)[0][0]
    hits = np.sum(y_true[mask] == dominant_label)
    correct_predictions += hits

print(f"Número de acertos: {correct_predictions} de {n_samples} pontos.")
print(f"Taxa de acerto: {(correct_predictions / n_samples):.2%}")

## Hierarchical

### Exercício 1: Implementação do Average Linkage

Complete a implementação da nossa classe `HierarchicalClustering` adicionando os métodos **Average Linkage** e **Ward Linkage**. Em seguida, teste todos os quatro métodos de ligação (single, complete, average, ward) no dataset simples (`X_simple`) e compare os resultados.

In [ ]:
class HierarchicalClustering:
    def __init__(self, linkage='single'):
        """
        Inicializa o algoritmo de clusterização hierárquica.
        """
        self.linkage = linkage
        self.merge_history = []
        self.distances = []

    def _calculate_distance_matrix(self, X):
        """
        Calcula a matriz de distâncias entre todos os pares de pontos.
        """
        n = len(X)
        dist_matrix = np.zeros((n, n))
        for i in range(n):
            for j in range(i+1, n):
                dist = np.linalg.norm(X[i] - X[j])
                dist_matrix[i, j] = dist
                dist_matrix[j, i] = dist

        # Para mais performance, use:
        # diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]
        # dist_matrix = np.linalg.norm(diff, axis=-1)

        return dist_matrix

    def _cluster_distance(self, cluster1, cluster2, X, dist_matrix):
        """
        Calcula a distância entre dois clusters baseado no critério de ligação.
        """
        if self.linkage == 'single':
            # Distância mínima entre qualquer par de pontos dos clusters
            min_dist = float('inf')
            for i in cluster1:
                for j in cluster2:
                    if dist_matrix[i, j] < min_dist:
                        min_dist = dist_matrix[i, j]
            return min_dist

        elif self.linkage == 'complete':
            # Distância máxima entre qualquer par de pontos dos clusters
            max_dist = 0
            for i in cluster1:
                for j in cluster2:
                    if dist_matrix[i, j] > max_dist:
                        max_dist = dist_matrix[i, j]
            return max_dist

        elif self.linkage == 'average':
          centroid_1 = np.mean(X[cluster1], axis=0)
          centroid_2 = np.mean(X[cluster2], axis=0)
          return np.linalg.norm(centroid_1 - centroid_2)
        else:
            print ("ERRor Metodo não suportado")

    def fit(self, X):
        """
        Executa o algoritmo de clusterização hierárquica aglomerativa.
        """
        n = len(X)

        # Inicializar cada ponto como um cluster
        clusters = [[i] for i in range(n)]

        # Calcular matriz de distâncias inicial
        dist_matrix = self._calculate_distance_matrix(X)

        self.merge_history = []
        self.distances = []

        step = 0
        print(f"Passo inicial: {len(clusters)} clusters individuais")
        print(f"Clusters: {clusters}\n")

        # Continuar até que reste apenas um cluster
        while len(clusters) > 1:
            # Encontrar o par de clusters mais próximo
            min_distance = float('inf')
            merge_i, merge_j = -1, -1

            for i in range(len(clusters)):
              for j in range(i+1, len(clusters)):
                    distance = self._cluster_distance(clusters[i], clusters[j], X, dist_matrix)
                    if distance < min_distance:
                        min_distance = distance
                        merge_i, merge_j = i, j

            # Combinar os clusters mais próximos
            new_cluster = clusters[merge_i] + clusters[merge_j]

            # Salvar informações da fusão
            self.merge_history.append((clusters[merge_i].copy(), clusters[merge_j].copy()))
            self.distances.append(min_distance)

            step += 1
            print(f"Passo {step}: Combinar clusters {clusters[merge_i]} e {clusters[merge_j]}")
            print(f"Distância: {min_distance:.3f}")

            # Remover os clusters antigos e adicionar o novo
            clusters = [clusters[k] for k in range(len(clusters)) if k != merge_i and k != merge_j]
            clusters.append(new_cluster)

            print(f"Clusters restantes: {clusters}\n")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_wine
from scipy.cluster.hierarchy import dendrogram, linkage
import pandas as pd
import seaborn as sns

np.random.seed(42)
X_simple = np.array([[1, 2], [1.5, 1.8], [5, 8], [8, 8], [1, 0.6], [9, 11]])

def plot_dendrogram(model, n_samples):
    linkage_matrix = []
    cluster_ids = {tuple([i]): i for i in range(n_samples)}
    current_id = n_samples

    for i in range(len(model.merge_history)):
        cluster_a, cluster_b = model.merge_history[i]
        dist = model.distances[i]

        id_a = cluster_ids[tuple(sorted(cluster_a))]
        id_b = cluster_ids[tuple(sorted(cluster_b))]

        new_cluster = sorted(cluster_a + cluster_b)
        sample_count = len(new_cluster)

        linkage_matrix.append([id_a, id_b, dist, sample_count])
        cluster_ids[tuple(new_cluster)] = current_id
        current_id += 1

    plt.figure(figsize=(10, 5))
    plt.title(f"Dendrograma - Ligação: {model.linkage}")
    dendrogram(np.array(linkage_matrix).astype(float))
    plt.ylabel("Distância")
    plt.show()


single_linkage = HierarchicalClustering(linkage = "single")
full_linkage = HierarchicalClustering(linkage ="complete")
average_linkage = HierarchicalClustering(linkage = "average")

single_linkage.fit(X_simple)
full_linkage.fit(X_simple)
average_linkage.fit(X_simple)


plot_dendrogram(single_linkage, len(X_simple))
plot_dendrogram(full_linkage, len(X_simple))
plot_dendrogram(average_linkage, len(X_simple))


### Exercício 2: Análise do Dataset Wine

Aplique a clusterização hierárquica do SciPy ao dataset Wine. Primeiro, você deve selecionar um bom par de features para visualização bidimensional, depois comparar diferentes métodos de ligação.

In [ ]:
from scipy.cluster.hierarchy import fcluster

wine = load_wine()
X_wine = wine.data
y_wine = wine.target

print("Dataset Wine:")
print(f"Shape: {X_wine.shape}")
print(f"Features: {wine.feature_names}")
print(f"Classes: {wine.target_names}")

#1 Análise das features para seleção
df = pd.DataFrame(X_wine, columns= wine.feature_names)
df["class"] = y_wine

sns.pairplot(df, hue="class")
plt.show()

#2 Seleção das duas melhores features
this_X = df[["od280/od315_of_diluted_wines", "proline"]].values

plt.scatter(df["od280/od315_of_diluted_wines"], df["proline"], c=df["class"], alpha=0.7)
plt.xlabel("od280/od315_of_diluted_wines")
plt.ylabel("proline")
plt.title("Dataset Wine - Relação entre proline e OD280/OD315 de vinhos diluidos")

plt.grid(True)
plt.show()

#3 Aplicação dos métodos de ligação e criação dos dendrogramas

methods = ['ward', 'complete', 'average', 'single']
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, method in enumerate(methods):
    #matriz
    linkage_matrix = linkage(this_X, method=method)

    dendrogram(linkage_matrix, ax=axes[0, i], no_labels=True)
    axes[0, i].set_title(f'Dendrograma - {method.capitalize()}')

    # Obter 3 clusters
    clusters = fcluster(linkage_matrix, 3, criterion='maxclust')

    # Plotar os clusters
    scatter = axes[1, i].scatter(this_X[:, 0], this_X[:, 1], c=clusters, s=50, alpha=0.7, cmap='viridis')
    axes[1, i].set_title(f'Clusters - {method.capitalize()}')

plt.tight_layout()
plt.show()

#4 Análise visual e determinação do melhor método
def calculate_purity_wine(y_true, y_pred):
    correct_predictions = 0
    n_samples = len(y_true)

    for cluster_id in np.unique(y_pred):
        mask = (y_pred == cluster_id)
        if np.sum(mask) > 0:
            dominant_label = mode(y_true[mask], keepdims=True)[0][0]
            correct_predictions += np.sum(y_true[mask] == dominant_label)

    return correct_predictions / n_samples

print("Comparação dos métodos de ligação no dataset Wine:")
print("="*50)

for i, method in enumerate(methods):
    linkage_matrix = linkage(this_X, method=method)
    clusters = fcluster(linkage_matrix, 3, criterion='maxclust')
    purity = calculate_purity_wine(y_wine, clusters)
    print(f"{method.capitalize():12} Linkage: {purity:.1%} de acertos")

### Exercício 3: Determinação do Número Ótimo de Clusters

Com base no melhor método de ligação identificado no Exercício 2, determine o número ótimo de clusters para o dataset Wine usando análise visual do dendrograma e validação com os rótulos verdadeiros.

In [ ]:
# Exercício 3: Determinação do Número Ótimo de Clusters

# 1. Use o melhor método do exercício anterior
best_method = 'ward'

# 2. Crie dendrograma com diferentes linhas de corte
def create_dendogram_with_cuts(X, method, n_clusters_list):
    linkage_matrix = linkage(X, method=method)

    plt.figure(figsize=(10, 6))
    dendrogram(linkage_matrix, no_labels=True)

    distances = linkage_matrix[:, 2]

    for n in n_clusters_list:
        if n <= 1: continue
        idx = len(distances) - n
        if idx < 0: continue
        height = (distances[idx] + distances[idx + 1]) / 2
        plt.axhline(y=height, color=np.random.rand(3,), linestyle='--', label=f'Corte para {n} Clusters (y={height:.2f})')

    plt.title(f'Dendrograma - {method.capitalize()} com Linhas de Corte')
    plt.xlabel('Pontos de Dados (Amostras do Wine)')
    plt.ylabel('Distância')
    plt.legend()
    plt.show()

# 3. Teste diferentes números de clusters
n_clusters_to_test = [2, 3, 4, 5]

print("Análise do número ótimo de clusters:")
print("=" * 40)

create_dendogram_with_cuts(this_X, best_method, n_clusters_to_test)

## DBSCAN

In [ ]:
!pip install dtaidistance

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN as SklearnDBSCAN
from scipy.spatial.distance import pdist, squareform
from scipy.stats import mode
import pandas as pd
from dtaidistance import dtw
from sklearn.neighbors import NearestNeighbors


class DBSCAN:
    def __init__(self, eps=0.5, min_pts=5, metric='euclidean'):
        """Inicializa o DBSCAN com os parâmetros eps e min_pts"""
        self.eps = eps
        self.min_pts = min_pts
        self.metric = metric
        self.labels_ = None
        self.core_samples_ = None
        self.n_clusters_ = 0

    def _calculate_distance_matrix(self, X):
        """Calcula a matriz de distâncias entre todos os pontos"""
        n = len(X)
        distances = np.zeros((n, n))
        if self.metric == 'euclidean':
            distances = np.linalg.norm(X[:, np.newaxis] - X, axis=2)
        elif self.metric == 'radial':
            norms = np.linalg.norm(X, axis=1)
            for i in range(n):
                for j in range(i + 1, n):
                    dist = np.abs(norms[i] - norms[j])
                    distances[i, j] = dist
                    distances[j, i] = dist
        elif self.metric == 'dtw':
            for i in range(n):
                for j in range(i + 1, n):
                    d = dtw.distance_fast(X[i].astype(np.double), X[j].astype(np.double))
                    distances[i, j] = distances[j, i] = d
        else:
            raise ValueError("Métrica não suportada")
        return distances

    def _get_neighbors(self, point_idx, distance_matrix):
        """Encontra todos os vizinhos dentro da distância eps"""
        return np.where(distance_matrix[point_idx] <= self.eps)[0]

    def _expand_cluster(self, point_idx, neighbors, cluster_id, distance_matrix, visited, labels):
        """Expande o cluster a partir do ponto inicial"""
        labels[point_idx] = cluster_id
        queue = neighbors.tolist()

        while queue:
            neighbor_idx = queue.pop(0)

            if not visited[neighbor_idx]:
                visited[neighbor_idx] = True
                neighbor_neighbors = self._get_neighbors(neighbor_idx, distance_matrix)

                if len(neighbor_neighbors) >= self.min_pts:
                    queue.extend(neighbor_neighbors)

            if labels[neighbor_idx] == -1:
                labels[neighbor_idx] = cluster_id

    def fit(self, X):
        """Executa o algoritmo DBSCAN"""
        n_points = len(X)
        visited = np.zeros(n_points, dtype=bool)
        cluster_id = 0
        self.labels_ = np.full(n_points, -1)  # -1 = ruído
        self.core_samples_ = []

        distance_matrix = self._calculate_distance_matrix(X)

        for point_idx in range(n_points):
            if visited[point_idx]:
                continue

            visited[point_idx] = True
            neighbors = self._get_neighbors(point_idx, distance_matrix)

            if len(neighbors) >= self.min_pts:   # core point
                self.core_samples_.append(point_idx)
                self._expand_cluster(point_idx, neighbors, cluster_id, distance_matrix, visited, self.labels_)
                cluster_id += 1

        self.core_samples_ = np.array(self.core_samples_)
        self.n_clusters_ = cluster_id
        return self

    def fit_predict(self, X):
        """Executa DBSCAN e retorna os labels"""
        self.fit(X)
        return self.labels_

### Exercício 1: Ajuste de Parâmetros

Utilizando os dados das três esferas concêntricas, construa o gráfico de K-Distance para diferentes valores de `min_pts` a fim de sugerir um intervalo adequado para `eps`, selecione os melhores valores para `min_pts` e `eps`, e, em seguida, visualize em 3D os clusters encontrados com cores distintas, discutindo criticamente como a escolha de `eps` e `min_samples` influenciou a separação das estruturas.

In [ ]:
def plot_k_distance(X, min_pts, title="K-Distance Plot"):
    """Plota o gráfico K-Distance usando sklearn.NearestNeighbors."""
    k = int(min_pts - 1)

    nn = NearestNeighbors(n_neighbors=k+1, metric="euclidean")
    nn.fit(X)
    distances, _ = nn.kneighbors(X)

    kth_distances = distances[:, k]
    k_distances_sorted = np.sort(kth_distances)

    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(range(len(k_distances_sorted)), k_distances_sorted, linewidth=2, label=f'{k}-distance')
    plt.xlabel("Pontos ordenados por distância")
    plt.ylabel(f"{k}-distance")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

def generate_concentric_spheres(radii=[3, 15], n_samples_per_sphere=1000, noise=0.2, random_state=42):
    """
    Gera pontos em 3 esferas concêntricas no espaço 3D.
    - radii: lista com os raios das esferas
    - n_samples_per_sphere: pontos em cada esfera
    - noise: variação radial para "espessura" da casca
    """
    rng = np.random.default_rng(random_state)
    X, y = [], []

    for i, r in enumerate(radii):
        # amostrar ângulos uniformemente
        phi = rng.uniform(0, 2*np.pi, n_samples_per_sphere)       # ângulo azimutal
        costheta = rng.uniform(-1, 1, n_samples_per_sphere)       # cos(theta)
        theta = np.arccos(costheta)                               # ângulo polar

        # raio com ruído
        rr = r + noise * rng.standard_normal(n_samples_per_sphere)

        # coordenadas cartesianas
        x = rr * np.sin(theta) * np.cos(phi)
        y_ = rr * np.sin(theta) * np.sin(phi)
        z = rr * np.cos(theta)

        X.append(np.vstack((x, y_, z)).T)
        y.append(np.full(n_samples_per_sphere, i))

    X = np.vstack(X)
    y = np.concatenate(y)
    return X, y

X_spheres, y_spheres = generate_concentric_spheres(radii=[3, 8, 12], n_samples_per_sphere=200, noise=0.4)

scaler = StandardScaler()
X_spheres = scaler.fit_transform(X_spheres)

In [ ]:
min_pts_values = [3, 5, 8, 10]

for min_pts in min_pts_values:
  plot_k_distance(X_spheres, min_pts, title="K-Distance Plot para Esferas Concêntricas (min_pts = " + str(min_pts) + ")")

In [ ]:
#Dado a analise dos cotovelos das curvas aprensentadas resolvir escolher os seguintes valores para eps
min_point_values = [3, 5, 8 ,10]
eps_values = [0.6 ,0.28, 0.32 , 0.38]

In [ ]:
import plotly.express as px
#função de plotar em 3d

def plot_3d_clusters(X, labels, title):
    fig = px.scatter_3d(
        x=X[:, 0],
        y=X[:, 1],
        z=X[:, 2],
        color=labels.astype(str), # Converter labels para string para cores categóricas
        color_continuous_scale=px.colors.qualitative.Vivid, # Usar uma escala de cores categórica
        title=title
    )
    fig.update_traces(marker=dict(size=3))
    fig.show()


for i in range(len(min_point_values)):
  dbscan = DBSCAN(eps=eps_values[i], min_pts=min_point_values[i])
  labels = dbscan.fit_predict(X_spheres)
  unique_labels = np.unique(labels)
  n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
  print(f"Resultados do DBSCAN (eps={eps_values[i]}, min_pts={min_point_values[i]}):")
  print(f"- Número de clusters encontrados: {n_clusters}")
  plot_3d_clusters(X_spheres,labels, f"Clusters Encontrados (eps={eps_values[i]}, min_pts={min_point_values[i]})")

### Análise dos Resultados do Exercício 1: Distancia Euclidiana

Nesta análise do dbscan com esferas concêntricas, a escolha de `eps` e `min_pts` foi crucial e demonstrou a sensibilidade do algoritmo à densidade e proximidade.

**Observações Principais:**

1.  **Impacto do `min_pts`:** Percebeu-se que, ao aumentar o `min_pts`, geralmente é necessário um `eps` maior para manter os clusters conectados. Um `min_pts` muito alto com um `eps` inadequado pode levar as camadas externas a serem tratadas como ruído, com apenas a esfera mais densa (núcleo) formando um cluster coeso.

2.  **Impacto do `eps`:**
    *   **`eps` muito alto:** Se o raio de vizinhança (`eps`) for excessivamente grande, ele pode abranger o espaço entre as esferas, fazendo com que múltiplas camadas concêntricas sejam percebidas como um único cluster. Isso resulta na perda da separação das estruturas.
    *   **`eps` muito baixo:** Por outro lado, um `eps` muito pequeno, especialmente com `min_pts` baixos, tende a fragmentar as esferas em muitos pequenos clusters ou a classificar muitos pontos como ruído, pois a densidade para formar um cluster não é alcançada uniformemente.

**Conclusão:**

A dificuldade reside em encontrar um `eps` que seja grande o suficiente para conectar todos os pontos dentro de uma mesma esfera, mas pequeno o bastante para evitar a conexão entre esferas distintas. Entretanto, idependente a heuristica utilizada, os resultados do agrupamento foram pouco precisos se comparado a distribuição dos dados.

Embora os resultados não sejam desejaveis, a combinação de atributos que retornou uma melhor clusterização dos dados foi `min_pts = 3` `eps = 0.6`

### Exercício 2: DBSCAN com Distância Radial

Usando os dados das três esferas concêntricas do exercício anterior, implemente a distância radial e utilize-a no DBSCAN, onde a distância entre dois pontos $x_i$ e $x_j$ é definida como $d_{\mathrm{radial}}(x_i, x_j) = \left| |x_i|_2 - |x_j|_2 \right|$; em seguida, plote o K-Distance radial para sugerir um valor adequado de `eps`, teste diferentes combinações de `eps` e `min_samples`, visualize em 3D os clusters obtidos e compare com o resultado utilizando a distância euclidiana, finalizando com uma análise breve sobre qual configuração apresentou melhor desempenho e por que a métrica radial ajuda nesse tipo de estrutura.

In [ ]:
def plot_k_distance_radial(X, min_pts, title="K-Distance Plot (Radial)"):
    """Plota o gráfico K-Distance usando a métrica radial."""
    k = int(min_pts - 1)

    #calcula a distancia do centro de cada ponto
    X_norms = np.linalg.norm(X, axis=1).reshape(-1, 1)


    nn = NearestNeighbors(n_neighbors=k+1, metric="euclidean")
    nn.fit(X_norms)
    distances, _ = nn.kneighbors(X_norms)

    kth_distances = distances[:, k]
    k_distances_sorted = np.sort(kth_distances)

    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(range(len(k_distances_sorted)), k_distances_sorted, linewidth=2, label=f'{k}-distance')
    plt.xlabel("Pontos ordenados por distância")
    plt.ylabel(f"{k}-distance (Radial)")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


min_point_values = [3, 5, 8 ,10]
for min_pts in min_point_values:
  plot_k_distance_radial(X_spheres, min_pts, title="K-Distance Plot para Esferas Concêntricas (radial) (min_pts = " + str(min_pts) + ")")

In [ ]:
# apos uma analise dos graficos de cotovelos setei nos seguintes valores de eps
min_point_values = [3, 5, 8 ,10]
eps_values =[0.013 ,0.020, 0.025,0.040]

In [ ]:
def plot_3d_clusters_radial(X, labels, title):
    fig = px.scatter_3d(
        x=X[:, 0],
        y=X[:, 1],
        z=X[:, 2],
        color=labels.astype(str), # Converter labels para string para cores categóricas
        color_continuous_scale=px.colors.qualitative.Vivid, # Usar uma escala de cores categórica
        title=title
    )
    fig.update_traces(marker=dict(size=3))
    fig.show()


for i in range(len(min_point_values)):
  dbscan = DBSCAN(eps=eps_values[i], min_pts=min_point_values[i],metric="radial")
  labels = dbscan.fit_predict(X_spheres)
  unique_labels = np.unique(labels)
  n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
  print(f"Resultados do DBSCAN (eps={eps_values[i]}, min_pts={min_point_values[i]}):")
  print(f"- Número de clusters encontrados: {n_clusters}")
  plot_3d_clusters(X_spheres,labels, f"Clusters Encontrados (eps={eps_values[i]}, min_pts={min_point_values[i]})")

### Análise dos Resultados do Exercício 2: Distância Radial

No geral, a implementação da métrica radial resultou em resultados de clustering muito superiores. A clareza das curvas do K-Distance, com cotovelos mais fáceis de identificar, já indicava um melhor desempenho.

Os resultados do dbscan com distância radial apresentaram uma clusterização mais acurada. A configuração de hiperparâmetros que demonstrou o **melhor desempenho foi `min_pts = 10` com `eps = 0.04`**.

A métrica radial se mostrou particularmente eficaz para essa estrutura de dados, pois mede diretamente a diferença de distância ao centro, o que é fundamental para separar anéis ou esferas de diferentes raios.

### Exercício 3: Detecção de Anomalias com DTW

O **DTW (Dynamic Time Warping)** mede a similaridade entre séries temporais mesmo quando elas estão defasadas no tempo ou evoluem com velocidades diferentes, realizando um alinhamento elástico entre seus pontos. Com isso, padrões semelhantes podem ser reconhecidos mesmo quando não estão perfeitamente sincronizados. Sua matriz de distâncias pode ser calculada, por exemplo, com:

```python
from dtaidistance import dtw

n = len(X)
D = np.zeros((n, n))

for i in range(n):
    for j in range(i + 1, n):
        dist = dtw.distance_fast(X[i], X[j])
        D[i, j] = D[j, i] = dist
```

Utilize o dataset de senóides com variação e anomalias simuladas para aplicar o DBSCAN com a métrica DTW, experimente diferentes valores de `eps` e `min_samples` até obter uma separação adequada entre séries normais e anômalas, e, ao final, plote todas as séries temporais destacando com uma cor as séries consideradas normais e com outra as séries detectadas como anomalias, isto é, aquelas com `label = -1`.

In [ ]:
def generate_time_series_dataset(n_series=50, length=100, noise=0.1, n_outliers=2, random_state=42):
    rng = np.random.default_rng(random_state)
    X, y = [], []
    t = np.linspace(0, 4*np.pi, length)

    # séries normais: senóide com amplitude e frequência ligeiramente diferentes
    for _ in range(n_series):
        amp = rng.uniform(0.8, 1.2)         # amplitude
        freq = rng.uniform(0.9, 1.1)        # frequência
        phase = rng.uniform(0, 0.5*np.pi)   # pequena defasagem
        series = amp * np.sin(freq * t + phase) + noise * rng.normal(size=length)
        X.append(series)
        y.append(0)  # normal

    # outliers: picos ou deslocamentos fortes
    for _ in range(n_outliers):
        amp = rng.uniform(1.5, 2.0)         # amplitude anômala
        freq = rng.uniform(1.2, 1.5)        # frequência anômala
        series = amp * np.sin(freq * t) + noise * rng.normal(size=length)
        if rng.random() < 0.5:
            series[length//2] += 3  # pico
        else:
            series += rng.normal(2.0, 0.5)  # deslocamento
        X.append(series)
        y.append(-1)  # anomalia

    return np.array(X), np.array(y)

X_series, y_series = generate_time_series_dataset()

plt.figure(figsize=(10,4))
for i in range(5):
    plt.plot(X_series[i], alpha=0.7, label="normal" if i==0 else "")
for i in range(-3,0):
    plt.plot(X_series[i], alpha=0.7, color="red", label="anomalia" if i==-1 else "")
plt.title("Séries temporais com anomalias")
plt.legend()
plt.show()

In [ ]:
eps_values = [1.0, 1.5, 2.0]
min_pts_values = [2, 3, 5]

fig, axes = plt.subplots(len(eps_values), len(min_pts_values), figsize=(18, 12))
fig.suptitle('Resultados DBSCAN + DTW para todas as combinações', fontsize=16)

for i, eps_val in enumerate(eps_values):
    for j, mp_val in enumerate(min_pts_values):
        model = DBSCAN(eps=eps_val, min_pts=mp_val, metric='dtw')
        current_labels = model.fit_predict(X_series)

        ax = axes[i, j]
        n_ano = np.sum(current_labels == -1)

        for idx in range(len(X_series)):
            color = 'red' if current_labels[idx] == -1 else 'tab:blue'
            alpha = 0.8 if current_labels[idx] == -1 else 0.1
            ax.plot(X_series[idx], color=color, alpha=alpha)

        ax.set_title(f"eps={eps_val}, min_pts={mp_val}\nAnomalias: {n_ano}")
        ax.grid(True, alpha=0.2)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## Clustering Metrics

In [ ]:
def silhouette_score(X, labels):
    """Calcula o Silhouette Score médio."""

    distances = cdist(X, X)
    unique_labels = np.unique(labels)
    unique_labels = unique_labels[unique_labels != -1]  # ignora outliers, se houver

    silhouette_values = []

    for i in range(len(X)):
        current_label = labels[i]

        # ignora pontos marcados como outlier
        if current_label == -1:
            continue

        # 1. a_i = distância média para pontos do mesmo cluster
        same_cluster = (labels == current_label)
        same_cluster[i] = False  # remove o próprio ponto

        if np.sum(same_cluster) == 0:
            a_i = 0
        else:
            a_i = np.mean(distances[i][same_cluster])

        # 2. b_i = menor distância média até outro cluster
        b_i = np.inf

        for other_label in unique_labels:
            if other_label == current_label:
                continue

            other_cluster = (labels == other_label)
            mean_distance = np.mean(distances[i][other_cluster])

            if mean_distance < b_i:
                b_i = mean_distance

        # 3. score de silhueta do ponto
        if max(a_i, b_i) == 0:
            s_i = 0
        else:
            s_i = (b_i - a_i) / max(a_i, b_i)

        silhouette_values.append(s_i)

    return np.mean(silhouette_values)

def davies_bouldin_score(X, labels):
    """Calcula o Davies-Bouldin Index."""

    # Remove rótulo -1, caso exista
    clusters = np.unique(labels)
    clusters = clusters[clusters != -1]
    n_clusters = len(clusters)

    # 1. Calcula o centróide de cada cluster
    centroids = []
    for c in clusters:
        points = X[labels == c]
        centroid = points.mean(axis=0)
        centroids.append(centroid)
    centroids = np.array(centroids)

    # 2. Calcula a dispersão de cada cluster
    dispersions = []
    for i, c in enumerate(clusters):
        points = X[labels == c]
        distances = np.linalg.norm(points - centroids[i], axis=1)
        dispersion = distances.mean()
        dispersions.append(dispersion)
    dispersions = np.array(dispersions)

    # 3. Para cada cluster i, acha o pior caso em relação aos outros clusters
    db_index = 0

    for i in range(n_clusters):
        worst_similarity = 0

        for j in range(n_clusters):
            if i == j:
                continue

            distance_between_centroids = np.linalg.norm(centroids[i] - centroids[j])
            similarity = (dispersions[i] + dispersions[j]) / distance_between_centroids

            if similarity > worst_similarity:
                worst_similarity = similarity

        db_index += worst_similarity

    # 4. Faz a média dos piores casos
    return db_index / n_clusters

def dunn_index(X, labels):
    """Calcula o Dunn Index para um conjunto de dados e seus rótulos."""

    unique_labels = np.unique(labels)
    pairwise_dists = squareform(pdist(X))

    # Máxima distância intra-cluster
    max_intra = 0
    for lab in unique_labels:
        idx = np.where(labels == lab)[0]
        if len(idx) > 1:
            d = np.max(pairwise_dists[np.ix_(idx, idx)])
            if d > max_intra:
                max_intra = d

    if max_intra == 0:
        return np.inf

    # Mínima distância inter-cluster
    min_inter = np.inf
    for i in range(len(unique_labels)):
        for j in range(i + 1, len(unique_labels)):
            idx_i = np.where(labels == unique_labels[i])[0]
            idx_j = np.where(labels == unique_labels[j])[0]
            d = np.min(pairwise_dists[np.ix_(idx_i, idx_j)])
            if d < min_inter:
                min_inter = d

    return min_inter / max_intra



### Exercício 1: Análise de Escolha de Hiperparâmetros

Altere os parâmetros iniciais dos algoritmos **K-Means**, **Hierárquico Aglomerativo** e **DBSCAN** (por exemplo, variando o número de clusters em `n_clusters` e os valores de `eps` e `min_samples`), execute os modelos nos datasets utilizados anteriormente, avalie os resultados com as métricas **Silhouette Score**, **Davies-Bouldin Index** e **Índice de Dunn**, e analise como as mudanças nos parâmetros influenciam a qualidade dos agrupamentos e o comportamento de cada métrica.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist, pdist, squareform
import pandas as pd

# Geração dos datasets
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.08, factor=0.4, random_state=42)
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)

# Normalização dos dados para que os algoritmos baseados em distância funcionem corretamente
X_blobs = StandardScaler().fit_transform(X_blobs)
X_circles = StandardScaler().fit_transform(X_circles)
X_moons = StandardScaler().fit_transform(X_moons)

datasets = {
    'Blobs': (X_blobs, y_blobs),
    'Circles': (X_circles, y_circles),
    'Moons': (X_moons, y_moons)
}

In [ ]:
# parametros
kmeans_params = {'n_clusters': 2, 'random_state': 42}
agglomerative_params = {'n_clusters': 6}
dbscan_params = {'eps': 0.5, 'min_samples': 6}

clustering_results = {}

for name, (X, y) in datasets.items():
    # K-Means
    kmeans = KMeans(**kmeans_params)
    kmeans_labels = kmeans.fit_predict(X)

    # Agglomerative Clustering
    agglomerative = AgglomerativeClustering(**agglomerative_params)
    agglomerative_labels = agglomerative.fit_predict(X)

    # DBSCAN
    dbscan = DBSCAN(**dbscan_params)
    dbscan_labels = dbscan.fit_predict(X)

    clustering_results[name] = {
        'K-Means': kmeans_labels,
        'Agglomerative': agglomerative_labels,
        'DBSCAN': dbscan_labels,
    }

In [ ]:
evaluation_results = []

for dataset_name, results in clustering_results.items():
    X, _ = datasets[dataset_name]
    for algo_name, labels in results.items():
        silhouette = silhouette_score(X, labels)
        dbi = davies_bouldin_score(X, labels)
        dunn = dunn_index(X, labels)

        evaluation_results.append({
            'Dataset': dataset_name,
            'Algorithm': algo_name,
            'Silhouette Score': silhouette,
            'Davies-Bouldin Index': dbi,
            'Dunn Index': dunn
        })
# Criar DataFrame para visualização
results_df = pd.DataFrame(evaluation_results)

# Dicionário de formatação apenas para colunas numéricas
formatter = {
    'Silhouette Score': '{:.3f}',
    'Davies-Bouldin Index': '{:.3f}',
    'Dunn Index': '{:.3f}'
}

# Aplicar o estilo e a formatação seletiva
results_df_styled = results_df.style.background_gradient(
    cmap='viridis', subset=['Silhouette Score', 'Dunn Index']
).background_gradient(
    cmap='viridis_r', subset=['Davies-Bouldin Index']
).format(formatter)

display(results_df_styled)


# Visualização dos resultados
fig, axes = plt.subplots(len(datasets), 3, figsize=(18, 15))
fig.suptitle('Resultados dos Algoritmos de Clustering', fontsize=16, fontweight='bold')

for i, (dataset_name, results) in enumerate(clustering_results.items()):
    X, _ = datasets[dataset_name]
    for j, (algo_name, labels) in enumerate(results.items()):
        ax = axes[i, j]
        # Pontos de ruído (-1) em preto para DBSCAN
        unique_labels = set(labels)
        colors = plt.cm.viridis(np.linspace(0, 1, len(unique_labels)))

        for k, col in zip(unique_labels, colors):
            if k == -1:
                col = [0, 0, 0, 1]  # Preto para ruído

            class_member_mask = (labels == k)
            xy = X[class_member_mask]
            ax.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
                    markeredgecolor='k', markersize=8, alpha=0.8)

        if i == 0:
            ax.set_title(algo_name, fontsize=14)
        if j == 0:
            ax.set_ylabel(dataset_name, fontsize=14, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# Analise das novas métricas apos a modificação dos parâmetros
Com os novos parâmetros `(KMeans=2, Agglomerative=6, DBSCAN eps=0.5)`, os resultados variaram conforme o dataset.
## Blobs:
K-Means teve desempenho inferior por reduzir de 4 para 2 clusters.
O Agglomerative apresentou queda no Silhouette e no Dunn Index, provavelmente devido à baixa dispersão e maior sobreposição entre clusters, causada pelo uso de 6 grupos em uma estrutura originalmente mais simples.
Já o DBSCAN obteve o melhor resultado, adaptando-se bem à densidade dos dados.

## Circles:
K-Means falhou por não lidar com formas não convexas, e o Agglomerative também não capturou corretamente a estrutura circular.
Ainda assim, as pontuações de ambos permaneceram praticamente iguais à análise anterior, o que é esperado dada a natureza do Silhouette Score que já não é bom para avaliar formas não convexas, o mesmo a respeito de dunn index já que em circles os cluesters basicamente se sobrepoem.

O DBSCAN retornou valores inválidos (NaN, 0, inf) porque, com os novos parâmetros, formou-se apenas um único cluster, inviabilizando o cálculo do silhouette e do dunn, além de levar à minimização de DBI.

## Moons:
O comportamento foi semelhante ao Circles.
K-Means e Agglomerative continuaram apresentando baixa qualidade, com valores de pontuação permanecendo praticamente os mesmos aos anteriores a modificação de parametros.
O DBSCAN repetiu o problema anterior, e a explicação segue a mesma: formação de um único cluster, como já mencionado no caso de Circles.

### Conclusão:
 Os parâmetros do DBSCAN performou abaixo do que eu esperava com a mudança de parametros, enquanto K-Means e Agglomerative mantiveram desempenho limitado pela própria natureza dos dados.

### Exercício 2: Avaliando Clustering do Dataset Wine

Aplique os algoritmos de agrupamento **K-Means**, **Hierárquico Aglomerativo** e **DBSCAN** ao conjunto de dados **Iris**, avalie os resultados utilizando as métricas **Silhouette Score**, **Davies-Bouldin Index** e **Índice de Dunn**, compare o desempenho dos métodos com base nessas métricas e discuta qual apresentou melhores resultados considerando as características dos dados e o comportamento de cada algoritmo.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X_iris = iris.data
y_iris = iris.target

X_iris = StandardScaler().fit_transform(X_iris)

datasets = {'Iris': (X_iris, y_iris)}

clustering_results = {}

for dataset_name, (X, _) in datasets.items():
    clustering_results[dataset_name] = {}

    clustering_results[dataset_name]['K-Means'] = KMeans(n_clusters=3, random_state=42).fit_predict(X)
    clustering_results[dataset_name]['Agglomerative'] = AgglomerativeClustering(n_clusters=3).fit_predict(X)
    clustering_results[dataset_name]['DBSCAN'] = DBSCAN(eps=0.5, min_samples=5).fit_predict(X)

evaluation_results = []

for dataset_name, results in clustering_results.items():
    X, _ = datasets[dataset_name]
    for algo_name, labels in results.items():
        silhouette = silhouette_score(X, labels)
        dbi = davies_bouldin_score(X, labels)
        dunn = dunn_index(X, labels)

        evaluation_results.append({
            'Dataset': dataset_name,
            'Algorithm': algo_name,
            'Silhouette Score': silhouette,
            'Davies-Bouldin Index': dbi,
            'Dunn Index': dunn
        })
# Criar DataFrame para visualização
results_df = pd.DataFrame(evaluation_results)

# Dicionário de formatação apenas para colunas numéricas
formatter = {
    'Silhouette Score': '{:.3f}',
    'Davies-Bouldin Index': '{:.3f}',
    'Dunn Index': '{:.3f}'
}

# Aplicar o estilo e a formatação seletiva
results_df_styled = results_df.style.background_gradient(
    cmap='viridis', subset=['Silhouette Score', 'Dunn Index']
).background_gradient(
    cmap='viridis_r', subset=['Davies-Bouldin Index']
).format(formatter)

display(results_df_styled)



fig, axes = plt.subplots(len(datasets), 3, figsize=(18, 15))
fig.suptitle('Resultados dos Algoritmos de Clustering', fontsize=16, fontweight='bold')

for i, (dataset_name, results) in enumerate(clustering_results.items()):
    X, _ = datasets[dataset_name]
    for j, (algo_name, labels) in enumerate(results.items()):

        ax = axes[j]
        # Pontos de ruído (-1) em preto para DBSCAN
        unique_labels = set(labels)
        colors = plt.cm.viridis(np.linspace(0, 1, len(unique_labels)))

        for k, col in zip(unique_labels, colors):
            if k == -1:
                col = [0, 0, 0, 1]  # Preto para ruído

            class_member_mask = (labels == k)
            xy = X[class_member_mask]
            ax.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
                    markeredgecolor='k', markersize=8, alpha=0.8)

        if i == 0:
            ax.set_title(algo_name, fontsize=14)
        if j == 0:
            ax.set_ylabel(dataset_name, fontsize=14, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# Analise das métricas no dataset íris:
## `DBSCAN (Silhouette score = 0,656 , DBI = 0,494, Dunn = 0,048)`
o DBSCAN foi oque obteve o melhor silhouette score e o menor DBI, indicando que os clusters encontrados dados os parametros de eps e min points são relativamente bem definidos.

## `Aglomerativo (Silhouette score = 0,447 , DBI = 0,803, Dunn = 0,098)`
O hierarquico aglomerativo apresenta o menor silhouette score e maior DBI, o que sugere um desempenho iferior aos demais metodos de clustering. Entretanto, o mesmo possui o maior valor de Dunn index o que indica uma melhor separação entre os clusters, embora o valor continue incrivelmente baixo dado a natureza caotica do dataset.

## `K-Means (Silhouette score = 0,480 , DBI = 0,789, Dunn = 0,053)`
K-means teve um desempenho que considero bom, mesmo sendo consideravelmente pior do que o do DBSCAN, isso é estava dentro de minhas previsões dado a natureza do k-means de desempenhar melhor com clusters esfericos.

### Conclusão
Minha analise mostrou que mesmo que dos 3 algoritmos de clusterização aquele que desempenhou melhor em separar os grupos, foi o DBSCAN que lidou com ruidos e conseguiu criar clusters bem definidos, coesos e dispersos. K-means e Hierarquico tambem performaram de maneira razoável, entretanto ambos apresentaram resultados inferiores.